# Lab - Compressão do JSON

In [1]:
from IPython.core.interactiveshell import InteractiveShell
InteractiveShell.ast_node_interactivity = "last_expr_or_assign"

In [2]:
import io, zipfile, json, pprint

## Ver o que há dentro sem extrair

In [3]:
with zipfile.ZipFile("Comiset23_Lab_Environment_Dataset.zip") as z:
    for info in z.infolist():
        print(info.filename, f"{info.file_size / 1e9:.2f}", "GB")

dataset_comillas2.json 159.73 GB


In [4]:
with zipfile.ZipFile("Comiset23_Lab_Environment_Dataset.zip") as z:
    print(f"{z.namelist()    = }")
    print(f"{z.namelist()[0] = }")

z.namelist()    = ['dataset_comillas2.json']
z.namelist()[0] = 'dataset_comillas2.json'


## Primeiro datapoint

In [5]:
with zipfile.ZipFile("Comiset23_Lab_Environment_Dataset.zip") as z:
    # pra inspecionar tem que abrir
    with z.open(z.namelist()[0]) as f:
        # usar TextIOWrapper pra ler em string ao invés de 'bytes'
        with io.TextIOWrapper(f, encoding='utf-8') as text_file:
            first_line = text_file.readline()
            datapoint = json.loads(first_line)
            # por algum motivo isso ordena os campos, é bom desativar
            pprint.pprint(datapoint, sort_dicts=False)

{'_index': 'logs-endpoint-winevent-additional-2022.11.16',
 '_type': '_doc',
 '_id': '8c0acd213cc3524fc717afbc33644d9c2eb6626c',
 '_score': 1,
 '_source': {'event_original_time': '2022-11-16T08:25:38.047Z',
             'etl_processed_time': '2022-11-16T08:28:16.386Z',
             'user_name': 'system',
             'PossibleCause': 'Unknown',
             'type': 'wineventlog',
             'process_id': 888,
             '@timestamp': '2022-11-16T08:25:38.047Z',
             'host_name': 'desktop-4pvps6e.phoenix.local',
             'level': 'error',
             'etl_host_agent_type': 'winlogbeat',
             'etl_pipeline': ['all-filter-0098',
                              'all-add_processed_timestamp',
                              'fingerprint-winlogbeats7',
                              'winlogbeat_7_and_above-field_nest_cleanup',
                              'winlogbeat_7_and_above-field_cleanups',
                              '1500',
                              '1522',


## DuckDB com configuração mínima

In [6]:
import pyarrow as pa, pyarrow.parquet as pq

In [7]:
import duckdb
from pathlib import Path

In [8]:
LAB_ORIGINAL_JSON = "Comiset23_Lab_Dataset/dataset_comillas2.json"
LAB_PARQUET_OUT  = "Parquet/lab_events.parquet"

'Parquet/lab_events.parquet'

In [9]:
con = duckdb.connect()
con.execute("PRAGMA threads=8")
con.execute("PRAGMA memory_limit='14GB'")
# con.execute("PRAGMA temp_directory='/mnt/d/duckdb_tmp'") # caso precise de spill

In [10]:
con.execute("SET enable_progress_bar = true;")
# Optional: Set it to print to terminal/stdout if widgets still fail
#con.execute("SET enable_progress_bar_print = true;")

## Compressão pra Parquet

In [11]:
con.execute(f"""
    COPY (
        SELECT * FROM read_ndjson(
            '{LAB_ORIGINAL_JSON}',
            auto_detect         = true,
            maximum_object_size = 104857600,
            ignore_errors       = true
        )
    )
    TO '{LAB_PARQUET_OUT}'
    (
        FORMAT         PARQUET,
        CODEC          'ZSTD', COMPRESSION_LEVEL 9
    )
""")

src  = Path(LAB_ORIGINAL_JSON).stat().st_size / 1e9
dest = Path(LAB_PARQUET_OUT).stat().st_size  / 1e9

print(f"JSON:    {src:.1f} GB")
print(f"Parquet: {dest:.1f} GB")
print(f"Razão:   {dest/src*100:.1f}%")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

JSON:    159.7 GB
Parquet: 2.6 GB
Razão:   1.6%


# Fim